In [1]:
import os
import json
from dotenv import load_dotenv
import anthropic
import yfinance as yf
from datetime import datetime

load_dotenv()
client = anthropic.Anthropic()
print("Setup complete")

Setup complete


In [2]:
def get_stock_price(ticker):
    """Fetch the most recent price for a stock ticker."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="5d")
        if data.empty:
            return f"Error: No data found for {ticker}"
        price = data["Close"].iloc[-1]
        date = data.index[-1].date()
        return f"{ticker} most recent price: ${price:.2f} (as of {date})"
    except Exception as e:
        return f"Error fetching {ticker}: {str(e)}"


def get_historical_return(ticker, years):
    """Compute total return of a stock over N years."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period=f"{years + 1}y")
        if data.empty or len(data) < 2:
            return f"Error: Insufficient data for {ticker}"
        start_price = data["Close"].iloc[0]
        end_price = data["Close"].iloc[-1]
        total_return = ((end_price - start_price) / start_price) * 100
        start_date = data.index[0].date()
        end_date = data.index[-1].date()
        return (f"{ticker} {years}-year return: {total_return:.2f}% "
                f"(${start_price:.2f} to ${end_price:.2f}, "
                f"{start_date} to {end_date})")
    except Exception as e:
        return f"Error computing return for {ticker}: {str(e)}"


def get_fundamentals(ticker):
    """
    Fetch the top 10 fundamental data points for value investors:
    1. Trailing P/E
    2. Forward P/E
    3. EV/EBITDA
    4. Price/Book
    5. Price/Free Cash Flow
    6. Operating Margin
    7. Return on Equity
    8. Revenue Growth (YoY)
    9. Debt/Equity
    10. Free Cash Flow Yield
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        def pct(val):
            """Convert decimal to percentage string."""
            if val is None:
                return "N/A"
            return f"{val * 100:.1f}%"

        def multiple(val):
            """Format a multiple with one decimal."""
            if val is None:
                return "N/A"
            return f"{val:.1f}x"

        def dollar_b(val):
            """Format a large dollar amount in billions."""
            if val is None:
                return "N/A"
            return f"${val / 1e9:.1f}B"

        # Raw values
        market_cap = info.get("marketCap")
        enterprise_value = info.get("enterpriseValue")
        ebitda = info.get("ebitda")
        free_cashflow = info.get("freeCashflow")
        trailing_pe = info.get("trailingPE")
        forward_pe = info.get("forwardPE")
        price_to_book = info.get("priceToBook")
        operating_margins = info.get("operatingMargins")
        return_on_equity = info.get("returnOnEquity")
        revenue_growth = info.get("revenueGrowth")
        debt_to_equity = info.get("debtToEquity")

        # Computed multiples
        ev_ebitda = None
        if enterprise_value and ebitda and ebitda > 0:
            ev_ebitda = enterprise_value / ebitda

        price_to_fcf = None
        if market_cap and free_cashflow and free_cashflow > 0:
            price_to_fcf = market_cap / free_cashflow

        fcf_yield = None
        if market_cap and free_cashflow and market_cap > 0:
            fcf_yield = free_cashflow / market_cap

        return {
            "company_name": info.get("longName", ticker),
            "sector": info.get("sector", "N/A"),
            "industry": info.get("industry", "N/A"),
            "market_cap": dollar_b(market_cap),
            "fifty_two_week_high": f"${info.get('fiftyTwoWeekHigh', 0):.2f}",
            "fifty_two_week_low": f"${info.get('fiftyTwoWeekLow', 0):.2f}",
            # The top 10
            "1_trailing_pe": multiple(trailing_pe),
            "2_forward_pe": multiple(forward_pe),
            "3_ev_ebitda": multiple(ev_ebitda),
            "4_price_to_book": multiple(price_to_book),
            "5_price_to_fcf": multiple(price_to_fcf),
            "6_operating_margin": pct(operating_margins),
            "7_return_on_equity": pct(return_on_equity),
            "8_revenue_growth_yoy": pct(revenue_growth),
            "9_debt_to_equity": multiple(debt_to_equity),
            "10_fcf_yield": pct(fcf_yield),
        }
    except Exception as e:
        return f"Error fetching fundamentals for {ticker}: {str(e)}"


# Quick test
print(json.dumps(get_fundamentals("AVGO"), indent=2))

{
  "company_name": "Broadcom Inc.",
  "sector": "Technology",
  "industry": "Semiconductors",
  "market_cap": "$2001.6B",
  "fifty_two_week_high": "$429.31",
  "fifty_two_week_low": "$184.02",
  "1_trailing_pe": "82.6x",
  "2_forward_pe": "23.3x",
  "3_ev_ebitda": "55.2x",
  "4_price_to_book": "25.1x",
  "5_price_to_fcf": "78.5x",
  "6_operating_margin": "44.9%",
  "7_return_on_equity": "33.4%",
  "8_revenue_growth_yoy": "29.5%",
  "9_debt_to_equity": "82.7x",
  "10_fcf_yield": "1.3%"
}


In [3]:
data_tools = [
    {
        "name": "get_stock_price",
        "description": "Fetch the most recent closing price for a stock ticker.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol (e.g. AAPL, NVDA)"
                }
            },
            "required": ["ticker"]
        }
    },
    {
        "name": "get_historical_return",
        "description": "Compute total percentage return of a stock over N years.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol"
                },
                "years": {
                    "type": "integer",
                    "description": "Number of years for the return calculation (1 or 5)"
                }
            },
            "required": ["ticker", "years"]
        }
    },
    {
        "name": "get_fundamentals",
        "description": (
            "Fetch the top 10 fundamental data points for value investors: "
            "trailing P/E, forward P/E, EV/EBITDA, price/book, price/FCF, "
            "operating margin, ROE, revenue growth, debt/equity, FCF yield."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol"
                }
            },
            "required": ["ticker"]
        }
    }
]

print(f"Tools registered: {[t['name'] for t in data_tools]}")

Tools registered: ['get_stock_price', 'get_historical_return', 'get_fundamentals']


In [4]:
def execute_tool(name, inputs):
    """Dispatch tool calls to the right Python function."""
    if name == "get_stock_price":
        return get_stock_price(**inputs)
    elif name == "get_historical_return":
        return get_historical_return(**inputs)
    elif name == "get_fundamentals":
        return get_fundamentals(**inputs)
    else:
        return f"Error: unknown tool '{name}'"


def gather_market_data(ticker):
    """
    Run the agent loop to systematically gather all market data.
    Returns the full conversation history and a summary string.
    """
    system_prompt = """You are a quantitative equity research analyst.
    Given a stock ticker, gather comprehensive market data using the available tools.
    Always fetch ALL of the following — do not skip any:
      1. Current price
      2. 1-year historical return
      3. 5-year historical return
      4. Fundamental data (the top 10 value investor metrics)
    Call all tools before responding. Be systematic and thorough."""

    messages = [{
        "role": "user",
        "content": (
            f"Gather all market data for {ticker}. "
            f"Fetch current price, 1-year return, 5-year return, and fundamentals."
        )
    }]

    print(f"Gathering market data for {ticker}...")

    while True:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=2048,
            system=system_prompt,
            tools=data_tools,
            messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            final_text = next(
                (b.text for b in response.content if b.type == "text"), ""
            )
            return messages, final_text

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  Calling {block.name}({block.input})")
                result = execute_tool(block.name, block.input)
                result_str = (
                    json.dumps(result) if isinstance(result, dict)
                    else str(result)
                )
                print(f"  → {result_str[:120]}...")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result_str
                })

        messages.append({"role": "user", "content": tool_results})

print("Data gathering loop defined")

Data gathering loop defined


In [5]:
research_report_tool = {
    "name": "produce_research_report",
    "description": (
        "Produce a concise, structured equity research report "
        "based on gathered market data. Use only actual data — "
        "do not estimate or fabricate any figures."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            # Identity
            "ticker": {"type": "string"},
            "company_name": {"type": "string"},
            "sector": {"type": "string"},
            "industry": {"type": "string"},
            # Price & performance
            "current_price": {
                "type": "number",
                "description": "Most recent price in USD"
            },
            "fifty_two_week_high": {"type": "string"},
            "fifty_two_week_low": {"type": "string"},
            "one_year_return_pct": {
                "type": "number",
                "description": "1-year return as a percentage (e.g. 45.2)"
            },
            "five_year_return_pct": {
                "type": "number",
                "description": "5-year return as a percentage"
            },
            "market_cap": {"type": "string"},
            # Top 10 value investor fundamentals
            "trailing_pe": {"type": "string"},
            "forward_pe": {"type": "string"},
            "ev_ebitda": {"type": "string"},
            "price_to_book": {"type": "string"},
            "price_to_fcf": {"type": "string"},
            "operating_margin": {"type": "string"},
            "return_on_equity": {"type": "string"},
            "revenue_growth_yoy": {"type": "string"},
            "debt_to_equity": {"type": "string"},
            "fcf_yield": {"type": "string"},
            # Analyst judgment
            "ai_rating": {
                "type": "string",
                "enum": ["Strong Buy", "Buy", "Hold", "Sell", "Strong Sell"],
                "description": (
                    "AI-generated rating based solely on the gathered data. "
                    "Strong Buy = compelling value + quality + growth. "
                    "Strong Sell = overvalued + deteriorating fundamentals."
                )
            },
            "ai_rating_rationale": {
                "type": "string",
                "description": (
                    "One sentence explaining the specific data points "
                    "that drove the AI rating. Be precise — cite actual numbers."
                )
            },
            "investment_thesis": {
                "type": "string",
                "description": (
                    "2-3 sentences. Bull case grounded in the actual data. "
                    "Concise and specific — no generic statements."
                )
            },
            "key_risks": {
                "type": "array",
                "items": {"type": "string"},
                "description": (
                    "Exactly 3 risks. Each should be one crisp sentence. "
                    "Ground them in the actual fundamental data where possible."
                )
            },
            "one_line_summary": {
                "type": "string",
                "description": (
                    "One punchy sentence. Lead with the rating rationale. "
                    "Under 20 words."
                )
            }
        },
        "required": [
            "ticker", "company_name", "sector", "industry",
            "current_price", "fifty_two_week_high", "fifty_two_week_low",
            "one_year_return_pct", "five_year_return_pct", "market_cap",
            "trailing_pe", "forward_pe", "ev_ebitda",
            "price_to_book", "price_to_fcf",
            "operating_margin", "return_on_equity",
            "revenue_growth_yoy", "debt_to_equity", "fcf_yield",
            "ai_rating", "ai_rating_rationale",
            "investment_thesis", "key_risks", "one_line_summary"
        ]
    }
}

print("Report schema defined")

Report schema defined


In [6]:
def generate_research_report(ticker, conversation_history, data_summary):
    """
    Force Claude to populate the structured report schema
    from the gathered conversation history.
    """
    system_prompt = """You are a senior equity research analyst.
    Produce a concise, data-driven research report.
    Rules:
    - Use ONLY actual numbers from the gathered data. No estimates.
    - The AI rating must be justified by specific metrics — cite them.
    - The thesis must be 2-3 sentences, specific, and grounded in the data.
    - Exactly 3 risks, each one crisp sentence.
    - The one-line summary must be under 20 words."""

    prompt = (
        f"Based on all market data gathered for {ticker}, "
        f"produce a structured research report.\n\n"
        f"Data summary: {data_summary}"
    )

    messages = conversation_history + [
        {"role": "user", "content": prompt}
    ]

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2048,
        system=system_prompt,
        tools=[research_report_tool],
        tool_choice={"type": "tool", "name": "produce_research_report"},
        messages=messages
    )

    tool_use = next(b for b in response.content if b.type == "tool_use")
    return tool_use.input

print("Report generator defined")

Report generator defined


In [7]:
def format_report(r):
    """
    Format the structured report dict into a clean, concise one-pager.
    r = report dict from generate_research_report()
    """
    W = 60
    div = "=" * W
    thin = "-" * W

    def row(label, value, width=22):
        """Print a labeled row, skip if value is N/A or missing."""
        val = value if value else "N/A"
        if val == "N/A":
            return ""
        return f"\n  {label:<{width}}{val}"

    output = f"""
{div}
  EQUITY RESEARCH — AI GENERATED
{div}
  {r['company_name']} ({r['ticker']})
  {r['sector']} | {r['industry']}
  Generated: {datetime.now().strftime('%B %d, %Y')}
{thin}
  AI RATING: {r['ai_rating'].upper()}
  {r['one_line_summary']}

  Rating rationale: {r['ai_rating_rationale']}
{div}
  PRICE & PERFORMANCE
{thin}{row('Current Price:', f"${r['current_price']:.2f}")}{row('52-Week Range:', f"{r['fifty_two_week_low']} – {r['fifty_two_week_high']}")}{row('1-Year Return:', f"{r['one_year_return_pct']:.1f}%")}{row('5-Year Return:', f"{r['five_year_return_pct']:.1f}%")}{row('Market Cap:', r['market_cap'])}
{div}
  TOP 10 VALUE INVESTOR FUNDAMENTALS
{thin}
  VALUATION
{row('1. Trailing P/E:', r['trailing_pe'])}{row('2. Forward P/E:', r['forward_pe'])}{row('3. EV / EBITDA:', r['ev_ebitda'])}{row('4. Price / Book:', r['price_to_book'])}{row('5. Price / FCF:', r['price_to_fcf'])}

  QUALITY & GROWTH
{row('6. Operating Margin:', r['operating_margin'])}{row('7. Return on Equity:', r['return_on_equity'])}{row('8. Revenue Growth:', r['revenue_growth_yoy'])}{row('9. Debt / Equity:', r['debt_to_equity'])}{row('10. FCF Yield:', r['fcf_yield'])}
{div}
  INVESTMENT THESIS
{thin}
  {r['investment_thesis']}
{div}
  KEY RISKS
{thin}"""

    for i, risk in enumerate(r['key_risks'], 1):
        output += f"\n  {i}. {risk}"

    output += f"""
{div}
  DISCLAIMER: AI-generated report for educational purposes only.
  Not investment advice. Verify all data before making decisions.
{div}"""

    return output

print("Formatter defined")

Formatter defined


In [8]:
def run_equity_research(ticker):
    """
    Full equity research pipeline:
    1. Gather market data via agent loop
    2. Generate structured report via forced tool use
    3. Format and print the one-pager
    4. Return the raw JSON for downstream use
    """
    ticker = ticker.upper()
    print(f"\n{'='*60}")
    print(f"  EQUITY RESEARCH: {ticker}")
    print(f"{'='*60}\n")

    # Step 1: Gather data
    history, summary = gather_market_data(ticker)
    print(f"\n  Data gathering complete.")

    # Step 2: Generate structured report
    print(f"  Generating report...")
    report = generate_research_report(ticker, history, summary)

    # Step 3: Format and display
    print(format_report(report))

    return report

print("Pipeline ready — run: run_equity_research('AVGO')")

Pipeline ready — run: run_equity_research('AVGO')


In [9]:
report = run_equity_research("AVGO")


  EQUITY RESEARCH: AVGO

Gathering market data for AVGO...
  Calling get_stock_price({'ticker': 'AVGO'})
  → AVGO most recent price: $422.76 (as of 2026-04-24)...
  Calling get_historical_return({'ticker': 'AVGO', 'years': 1})
  → AVGO 1-year return: 233.27% ($126.85 to $422.76, 2024-04-25 to 2026-04-24)...
  Calling get_historical_return({'ticker': 'AVGO', 'years': 5})
  → AVGO 5-year return: 1703.70% ($23.44 to $422.76, 2020-04-27 to 2026-04-24)...
  Calling get_fundamentals({'ticker': 'AVGO'})
  → {"company_name": "Broadcom Inc.", "sector": "Technology", "industry": "Semiconductors", "market_cap": "$2001.6B", "fifty...

  Data gathering complete.
  Generating report...

  EQUITY RESEARCH — AI GENERATED
  Broadcom Inc. (AVGO)
  Technology | Semiconductors
  Generated: April 26, 2026
------------------------------------------------------------
  AI RATING: HOLD
  Exceptional margins and growth justify premium, but 82.6x P/E requires flawless execution at 82.7x leverage.

  Rating rat